# 01 문서를 통째로 넣기

보내는 양이 문서 수에 비례한다. 비용·지연·정확도 세 축의 기준선.


In [ ]:
from pathlib import Path  # 경로를 문자열 대신 객체로 다룬다
import os  # 환경변수(OPENAI_API_KEY)를 넣기 위해 쓴다

# 수업 코드는 키가 이미 있는 상태를 가정한다. 이 노트북은 .env를 직접 읽는다.
for _env in (Path("../.env"), Path("../../c3-api/.env")):  # 프로젝트 루트, 옆 폴더 순으로 찾는다
    if not _env.is_file():  # 파일이 없으면 다음 후보
        continue  # 있는 파일만 읽는다
    for _line in _env.read_text(encoding="utf-8").splitlines():  # .env를 한 줄씩
        _line = _line.strip()  # 앞뒤 공백 제거
        if not _line or _line.startswith("#") or "=" not in _line:  # 빈 줄·주석·형식 아닌 줄
            continue  # 건너뛴다
        _k, _v = _line.split("=", 1)  # KEY=VALUE 로 나눈다
        os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))  # 이미 있으면 덮지 않는다


In [ ]:
from pathlib import Path  # 윈도우/리눅스 모두에서 같은 방식으로 폴더를 가리킨다

CORPUS = Path("../corpus")  # day02 에서 한 단계 위 = 프로젝트의 corpus (day01/corpus 링크)
if not (CORPUS / "text").is_dir():  # VS Code가 루트에서 실행하면 ../corpus 가 빗나간다
    _here = Path.cwd().resolve()  # 지금 작업 폴더
    for _p in [_here, *_here.parents]:  # 위로 올라가며 찾는다
        if (_p / "corpus" / "text").is_dir():  # corpus/text 가 있으면 그게 코퍼스다
            CORPUS = _p / "corpus"  # 찾은 경로로 고친다
            break  # 더 위는 보지 않는다
TEXT = CORPUS / "text"  # 어제 뽑아 둔 글자 파일들이 있는 폴더
print("CORPUS", CORPUS.resolve())  # 실제로 어디를 보는지 확인한다

from openai import OpenAI  # 생성 모델 호출용 공식 SDK
import tiktoken  # 토큰 수를 재는 라이브러리
import time  # 왕복이 얼마나 걸리는지 재기 위해 쓴다

# 글자수와 토큰 수 확인
docs = {p.stem: p.read_text(encoding="utf-8") for p in sorted(TEXT.iterdir())}  # 파일 이름(확장자 제외) → 본문
enc = tiktoken.get_encoding("o200k_base")  # gpt-4o / gpt-5 계열이 쓰는 토큰 규칙

for name, body in docs.items():  # 문서마다
    print(f"{name:20s} {len(body):>8,}자  {len(enc.encode(body)):>7,}토큰")  # 글자 수와 토큰 수를 나란히 찍는다

whole = "\n\n".join(f"[{name}]\n{body}" for name, body in docs.items())  # 다섯 문서를 한 덩어리로 붙인다
print("-" * 46)  # 구분선
print(f"{'합계':20s} {len(whole):>8,}자  {len(enc.encode(whole)):>7,}토큰")  # 통째로 넣을 때 나가는 양


아래는 문서 전체를 프롬프트에 붙여 한 번에 묻는다. RAG가 아니다. 기준선이다.


In [ ]:
client = OpenAI()  # API 키는 환경변수 OPENAI_API_KEY 를 읽는다
API_MODEL = "gpt-5.6-luna"  # 수업에서 쓰는 생성 모델 이름

QUESTION = "개인정보 보호법에서 '개인정보처리자'는 어떻게 정의되어 있나? 정의 조항의 문구를 그대로 인용해줘."  # 답이 법에 있는 질문

prompt_whole = f"아래 문서에서만 근거를 찾아 답해.\n\n{whole}\n\n[질문]\n{QUESTION}"  # 문서+질문을 한 문자열로 만든다

t0 = time.time()  # 보내기 직전 시각
r_whole = client.responses.create(model=API_MODEL, input=prompt_whole)  # 통째로 넣은 요청
elapsed_whole = time.time() - t0  # 끝난 시각과의 차이 = 지연

d0 = r_whole.usage.input_tokens_details  # 입력 토큰의 세부(캐시 포함)
print(r_whole.output_text)  # 모델이 말한 답
print()  # 빈 줄
print(f"걸린 시간  : {elapsed_whole:.1f}초")  # 지연
print(f"입력 토큰  : {r_whole.usage.input_tokens:,}")  # 이번에 청구되는 입력량
print(f"캐시 읽기  : {d0.cached_tokens:,}")  # 캐시에서 읽힌 토큰. 있어도 문서를 붙인 사실은 같다
